In [1]:
%load_ext autoreload
%autoreload 2

In [2]:

import ex1_parametric
import os
import CoRT_builder
import SI_CoRT
import datetime
import json
import time

In [3]:
n_source = 80
p = 200
K = 5
Ka = 2
h = 15
alpha = 0.05
T = 3
s_len = 10
s_vector = [0.5] * s_len
iteration = 1000

n_target_list = [60, 70]

for n_target in n_target_list:
    print(f"-- Processing n_target {n_target} --")
    para_results_storage = []
    CoRT_model = CoRT_builder.CoRT()

    cnt1 = 0
    cnt2 = 0
    cnt3 = 0
    cnt4 = 0

    start = time.time()
    for i in range(iteration):
        target_data, source_data = CoRT_model.gen_data(n_target, n_source, p, K, Ka, h, s_vector, s_len, "AR")
        result_dict = SI_CoRT.SI_parametric(n_target, p, K, target_data, source_data, T, s_len)

        if result_dict != None:
            cnt1 += (result_dict['is_signal'] == True)
            cnt2 += (result_dict['is_signal'] == False)
            cnt3 += (result_dict['is_signal'] == True and result_dict['p_value'] <= alpha)
            cnt4 += (result_dict['is_signal'] == False and result_dict['p_value'] <= alpha)
            
        if i % 1 == 0:
            print(f"is_signal : {result_dict['is_signal']}, p_values[{i}]: {result_dict['p_value']}")
            print(f"FPR: {cnt4 / cnt2}, TPR: {cnt3 / cnt1}")
            print(f"is_not_signal: {int(cnt2), int(cnt4)}")
            print(f"is_signal: {int(cnt1), int(cnt3)}")
            average = (time.time() - start) / (i+1)
            print(f"Time/iter: {average:.4f}s")
            print("===========================================================================")
                
            para_results_storage.append(result_dict)
    is_signal_cases = [r for r in para_results_storage if r['is_signal']]
    not_signal_cases = [r for r in para_results_storage if not r['is_signal']]

    false_positives = sum(1 for c in not_signal_cases if c['p_value'] <= alpha)
    fpr = false_positives / len(not_signal_cases) if len(not_signal_cases) != 0 else -1
    true_positives = sum(1 for r in is_signal_cases if r['p_value'] <= alpha) 
    tpr = true_positives / len(is_signal_cases) if len(is_signal_cases) != 0 else -1

    configs = {
        "n_source": n_source,
        "p":  p,
        "K": K,
        "Ka": Ka,
        "h": h,
        "alpha": alpha,
        "T":  T,
        "s_len": s_len,
        "s_vector": s_vector,
        "n_target": n_target,
        "iteration": iteration,
    }

    results = {
        "configs": configs,
        "time": datetime.datetime.now().ctime(),
        "fpr": fpr,
        "tpr": tpr
    }

    filename = 'records/parametric_vis_records.json'
    if os.path.exists(filename):
        with open(filename, 'r') as f:
            try:
                records = json.load(f)
            except Exception as e:
                print(e)
                records = []
    else:
        records = []

    records.append(results)

    with open(filename, 'w') as f:
        json.dump(records, f, indent=4)

-- Processing n_target 60 --
is_signal : True, p_values[0]: 0.010556444382224806
FPR: nan, TPR: 1.0
is_not_signal: (0, 0)
is_signal: (1, 1)
Time/iter: 9.8597s
is_signal : True, p_values[1]: 0.0004203680455112657
FPR: nan, TPR: 1.0
is_not_signal: (0, 0)
is_signal: (2, 2)
Time/iter: 10.7900s
is_signal : True, p_values[2]: 0.0022417628554658453
FPR: nan, TPR: 1.0
is_not_signal: (0, 0)
is_signal: (3, 3)
Time/iter: 10.5295s
is_signal : False, p_values[3]: 0.3734343799249158
FPR: 0.0, TPR: 1.0
is_not_signal: (1, 0)
is_signal: (3, 3)
Time/iter: 9.7906s
is_signal : True, p_values[4]: 0.0002956356835843721
FPR: 0.0, TPR: 1.0
is_not_signal: (1, 0)
is_signal: (4, 4)
Time/iter: 9.9316s
is_signal : True, p_values[5]: 4.042884691468629e-07
FPR: 0.0, TPR: 1.0
is_not_signal: (1, 0)
is_signal: (5, 5)
Time/iter: 9.7674s
is_signal : True, p_values[6]: 5.538855054165737e-06
FPR: 0.0, TPR: 1.0
is_not_signal: (1, 0)
is_signal: (6, 6)
Time/iter: 9.5960s
is_signal : True, p_values[7]: 0.002275323391268902
FPR

KeyboardInterrupt: 

In [ ]:
result_dict

{'p_value': None, 'is_signal': np.True_, 'feature_idx': np.int64(3)}